# Does Qwen3.6-35B-A3B plan its final answer? — Forward-planning detection

## Research question

Anthropic (2025) showed Claude 3.5 Haiku **plans rhyming words BEFORE writing poetry lines** — features for the final word activate at the newline token, before the line begins. [On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html)

**They did NOT test reasoning.** Explicitly stated as open question in the paper.

**Our hypothesis**: Qwen3.6-35B-A3B (hybrid MoE+GDN+Gated-Attn) planning in **multiple-choice reasoning** — does the activation at the FIRST 100 tokens of the response already encode the final `\boxed{letter}` answer?

## Why this matters

1. **First planning study on reasoning tasks** (Anthropic did poetry only)
2. **First on hybrid MoE architecture** (they tested dense Claude 3.5 Haiku)
3. **Uses our Stage B corpus** — 1000 labeled rollouts with per-token L11/L17/L23 activations (unique dataset)
4. **Zero GPU required** — analysis is CPU-only logistic regression

## Method

For each rollout (prompt + generated response + predicted letter):
1. Extract activations of the **first N tokens** of the response (early window)
2. Mean-pool → single feature vector per rollout per layer
3. Train multi-class classifier: early_activations → predicted_letter (A-J)
4. Compare vs chance (1/n_options ≈ 11% for 9-option MCQ)
5. Sweep over early window sizes: 10, 25, 50, 100, 150 tokens
6. Sweep over layers: L11, L17, L23

**Planning detected** if: early pool (≤100 tokens) predicts final letter significantly above chance.

## Expected outcomes

| Finding | Interpretation |
|---|---|
| Accuracy at 50 tokens ≈ accuracy at 150 tokens | **Strong planning** — model knows answer by token 50 |
| Accuracy rises steeply from 10 → 150 | **Progressive reasoning** — answer crystallizes over time |
| Accuracy ≈ chance | **No planning** — answer emerges only at boxed commit |
| Correct rollouts show stronger early signal than wrong | **Planning is load-bearing** — wrong rollouts are reasoning failures |

## Cell 1 — Install + imports

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub==1.5.0', 'safetensors',
                'scikit-learn', 'scipy', 'matplotlib', 'seaborn', 'tqdm',
                'hf_transfer'], check=True)

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import os, re, json, time, pickle
from pathlib import Path
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/planning')
OUT.mkdir(exist_ok=True)
print(f'workspace: {OUT}')

## Cell 2 — CFG

In [ ]:
CFG = dict(
    stage_b_repo='caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    layers=[11, 17, 23],
    # Windows de tokens pra testar (primeiros N)
    early_windows=[10, 25, 50, 100, 150],
    # Baseline: janela do fim (sinal máximo)
    late_window=50,
    # Split
    test_frac=0.2,
    seed=42,
    # Minimum response length to include rollout
    min_response_len=200,  # need enough tokens for 150-window + 50-late-window
)
print(json.dumps(CFG, indent=2))

## Cell 3 — Download Stage B corpus

~25 GB, ~15 min com hf_transfer. Colab free tem ~80 GB disk, cabe.

In [ ]:
from huggingface_hub import snapshot_download

corpus_path = snapshot_download(
    CFG['stage_b_repo'], repo_type='dataset',
    cache_dir='/content/cache')
shard_dir = Path(corpus_path) / 'shards'
shards = sorted(shard_dir.glob('q*.safetensors'))
print(f'\u2705 {len(shards)} shards downloaded')
print(f'Total size: {sum(s.stat().st_size for s in shards)/1e9:.1f} GB')

## Cell 4 — Extract early + late window activations

Streamed through shards, keeps only windows needed (~2 GB in RAM total).

In [ ]:
from safetensors import safe_open
from tqdm.auto import tqdm

# Accumulators: for each window size, for each layer, list of pooled vectors per rollout
# Shape at end: [n_windows, n_layers, n_rollouts, d_model]
early_pools = {w: {li: [] for li in CFG['layers']} for w in CFG['early_windows']}
late_pools = {li: [] for li in CFG['layers']}  # last 50 tokens
labels_letter = []  # predicted letter (A-J)
labels_correct = []  # was it correct?
meta_info = []  # extra per rollout

max_window = max(CFG['early_windows'])

for shard in tqdm(shards, desc='extract'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offsets_all = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        n_options = len(json.loads(meta['options']))
        acts_by_layer = {li: f.get_tensor(f'acts_L{li}') for li in CFG['layers']}
        offsets_by_layer = {li: offsets_all[f'L{li}'] for li in CFG['layers']}
        for r_idx, r in enumerate(rollouts):
            L = r['response_len']
            pred = r['pred']
            if pred is None or L < CFG['min_response_len']:
                continue
            for li in CFG['layers']:
                acts = acts_by_layer[li]
                start = offsets_by_layer[li][r_idx]
                end = offsets_by_layer[li][r_idx+1]
                rollout_acts = acts[start:end].float().numpy()  # [L, d_model]
                for w in CFG['early_windows']:
                    pool = rollout_acts[:w].mean(axis=0)
                    early_pools[w][li].append(pool)
                late_pools[li].append(rollout_acts[-CFG['late_window']:].mean(axis=0))
            labels_letter.append(pred)
            labels_correct.append(int(r['correct']))
            meta_info.append(dict(
                question_idx=int(meta['question_global_idx']),
                rollout_id=r['rollout_id'],
                response_len=L,
                n_options=n_options,
                discipline=meta.get('discipline', ''),
            ))

# Stack into numpy arrays
for w in CFG['early_windows']:
    for li in CFG['layers']:
        early_pools[w][li] = np.stack(early_pools[w][li])
for li in CFG['layers']:
    late_pools[li] = np.stack(late_pools[li])
labels_letter = np.array(labels_letter)
labels_correct = np.array(labels_correct)

print(f'\n\u2705 Extracted {len(labels_letter)} rollouts')
print(f'Shape early[100][L11]: {early_pools[100][11].shape}')
print(f'Shape late[L11]: {late_pools[11].shape}')
print(f'Letter distribution: {Counter(labels_letter)}')
print(f'Correct: {labels_correct.sum()}/{len(labels_correct)} ({100*labels_correct.mean():.1f}%)')

# Save for re-use (don't need to re-extract)
np.savez_compressed(OUT / 'activations.npz',
                    labels_letter=labels_letter, labels_correct=labels_correct,
                    **{f'early_w{w}_L{li}': early_pools[w][li]
                       for w in CFG['early_windows'] for li in CFG['layers']},
                    **{f'late_L{li}': late_pools[li] for li in CFG['layers']})
print(f'\u2705 saved {OUT}/activations.npz')

## Cell 5 — Train classifiers for each layer × window

Multi-class (letters A-J). Chance rate depends on n_options per question. Report macro accuracy + per-class.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, top_k_accuracy_score, confusion_matrix

# Encode letters
letter_to_int = {l: i for i, l in enumerate(sorted(set(labels_letter)))}
y = np.array([letter_to_int[l] for l in labels_letter])
n_classes = len(letter_to_int)
chance = 1.0 / n_classes
print(f'Classes: {list(letter_to_int.keys())} ({n_classes} total)  | chance = {chance:.3f}')

# Single stratified split used across all configs
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=CFG['test_frac'],
                                        stratify=y, random_state=CFG['seed'])
print(f'Train: {len(idx_train)}  Test: {len(idx_test)}')

def eval_config(X_all, name):
    X_tr, X_te = X_all[idx_train], X_all[idx_test]
    y_tr, y_te = y[idx_train], y[idx_test]
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_te_s = scaler.transform(X_te)
    pca = PCA(n_components=min(128, X_tr_s.shape[1]), random_state=42).fit(X_tr_s)
    Xt = pca.transform(X_tr_s)
    Xv = pca.transform(X_te_s)
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42, multi_class='multinomial').fit(Xt, y_tr)
    y_pred = lr.predict(Xv)
    y_proba = lr.predict_proba(Xv)
    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    top2 = top_k_accuracy_score(y_te, y_proba, k=2, labels=list(range(n_classes)))
    return dict(name=name, acc=acc, bal_acc=bal_acc, top2=top2)

# Evaluate all configs
results = []
for w in CFG['early_windows']:
    for li in CFG['layers']:
        r = eval_config(early_pools[w][li], f'early_w{w}_L{li}')
        r['window'] = w; r['layer'] = li; r['type'] = 'early'
        results.append(r)
for li in CFG['layers']:
    r = eval_config(late_pools[li], f'late_L{li}')
    r['window'] = CFG['late_window']; r['layer'] = li; r['type'] = 'late'
    results.append(r)

# Print table
print(f'\n{"="*70}')
print(f'{"Config":<25} {"Accuracy":>10} {"BalAcc":>10} {"Top-2":>10}')
print(f'{"-"*70}')
print(f'{"CHANCE":<25} {chance:>10.3f} {chance:>10.3f} {2/n_classes:>10.3f}')
print(f'{"-"*70}')
for r in sorted(results, key=lambda r: (-r['window'] if r['type']=='late' else r['window'], r['layer'])):
    tag = '\u2b50' if r['acc'] > chance * 2 else ('\u2705' if r['acc'] > chance * 1.5 else ('\u26a0\ufe0f' if r['acc'] > chance else '\u274c'))
    print(f'{r["name"]:<25} {r["acc"]:>10.3f} {r["bal_acc"]:>10.3f} {r["top2"]:>10.3f}  {tag}')

with open(OUT / 'planning_results.json', 'w') as f:
    json.dump(dict(chance=chance, n_classes=n_classes, results=results), f, indent=2)

## Cell 6 — Plot: planning detection curves

Acc vs token window, one line per layer. If planning exists, early windows already predict at > 2x chance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: accuracy vs window size by layer
ax = axes[0]
colors = {11: '#3498db', 17: '#9b59b6', 23: '#2ecc71'}
for li in CFG['layers']:
    xs = CFG['early_windows']
    ys = [[r['acc'] for r in results if r['type']=='early' and r['window']==w and r['layer']==li][0] for w in xs]
    ax.plot(xs, ys, 'o-', color=colors[li], lw=2, label=f'L{li} (early)')
    # Late baseline
    late = [r['acc'] for r in results if r['type']=='late' and r['layer']==li][0]
    ax.axhline(late, ls=':', color=colors[li], alpha=0.5, label=f'L{li} late (last 50 tok)')
ax.axhline(chance, ls='--', color='red', alpha=0.7, label=f'chance ({chance:.2f})')
ax.axhline(chance*2, ls='--', color='orange', alpha=0.5, label=f'2×chance ({2*chance:.2f})')
ax.set_xlabel('Early window (first N response tokens)')
ax.set_ylabel('Classifier accuracy (predict letter A-J)')
ax.set_title('Planning detection: early activation → final answer')
ax.legend(loc='best', fontsize=9)
ax.set_xscale('log')

# Right: balanced accuracy version (robust to class imbalance)
ax = axes[1]
for li in CFG['layers']:
    xs = CFG['early_windows']
    ys = [[r['bal_acc'] for r in results if r['type']=='early' and r['window']==w and r['layer']==li][0] for w in xs]
    ax.plot(xs, ys, 'o-', color=colors[li], lw=2, label=f'L{li}')
    late = [r['bal_acc'] for r in results if r['type']=='late' and r['layer']==li][0]
    ax.axhline(late, ls=':', color=colors[li], alpha=0.5)
ax.axhline(chance, ls='--', color='red', alpha=0.7, label='chance')
ax.set_xlabel('Early window (log scale)')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Planning detection (balanced accuracy)')
ax.legend(loc='best', fontsize=9)
ax.set_xscale('log')

plt.tight_layout()
plt.savefig(OUT / 'planning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\u2705 saved {OUT}/planning_curves.png')

## Cell 7 — Correct vs Wrong: does planning differ?

If planning is load-bearing for reasoning, correct-answer rollouts should have stronger early planning signal than wrong-answer rollouts.

Compare: classifier accuracy on correct-rollouts subset vs wrong-rollouts subset.

In [ ]:
# For best early window (say, first 100 tokens) at best layer, compare subsets
best_config = max([r for r in results if r['type']=='early'], key=lambda r: r['bal_acc'])
print(f'Best early config: {best_config["name"]} acc={best_config["acc"]:.3f}')

w_best = best_config['window']
li_best = best_config['layer']
X = early_pools[w_best][li_best]

idx_tr, idx_te = idx_train, idx_test
is_correct_te = labels_correct[idx_te]

# Fit on ALL train (preserving the classifier), evaluate separately on correct vs wrong test
scaler = StandardScaler().fit(X[idx_tr])
pca = PCA(n_components=128, random_state=42).fit(scaler.transform(X[idx_tr]))
Xt = pca.transform(scaler.transform(X[idx_tr]))
Xv = pca.transform(scaler.transform(X[idx_te]))
lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42, multi_class='multinomial').fit(Xt, y[idx_tr])
y_pred = lr.predict(Xv)

acc_correct = accuracy_score(y[idx_te][is_correct_te==1], y_pred[is_correct_te==1]) if is_correct_te.sum() > 0 else None
acc_wrong = accuracy_score(y[idx_te][is_correct_te==0], y_pred[is_correct_te==0]) if (is_correct_te==0).sum() > 0 else None

print(f'\nBest config: {best_config["name"]}')
print(f'  Overall accuracy: {best_config["acc"]:.3f}')
print(f'  On CORRECT rollouts (n={is_correct_te.sum()}): {acc_correct:.3f}' if acc_correct else '  no correct in test')
print(f'  On WRONG rollouts (n={(is_correct_te==0).sum()}): {acc_wrong:.3f}' if acc_wrong else '  no wrong in test')
if acc_correct and acc_wrong:
    gap = acc_correct - acc_wrong
    print(f'  Δ(correct - wrong): {gap:+.3f}')
    if gap > 0.10:
        print('  \u2b50 Planning is LOAD-BEARING: correct rollouts plan better')
    elif gap > 0.02:
        print('  \u2705 weak signal — correct plans slightly better')
    elif abs(gap) <= 0.02:
        print('  \u26a0\ufe0f same planning strength — answer is determined early regardless of outcome')
    else:
        print('  \u274c wrong rollouts plan better??? unexpected')

## Cell 8 — Confusion matrix at best early config

Which letters is the classifier confusing? Clean vs diagonal-heavy.

In [ ]:
int_to_letter = {i: l for l, i in letter_to_int.items()}
labels_list = [int_to_letter[i] for i in range(n_classes)]

cm = confusion_matrix(y[idx_te], y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels_list, yticklabels=labels_list, ax=ax, cbar=True)
ax.set_xlabel('Predicted')
ax.set_ylabel('True (actually what model answered)')
ax.set_title(f'Confusion — best config: {best_config["name"]} (acc={best_config["acc"]:.3f})')
plt.tight_layout()
plt.savefig(OUT / 'planning_confusion.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\u2705 saved {OUT}/planning_confusion.png')

## Cell 9 — Verdict + report

Decision rules:

- **Strong planning** (best early acc > 2× chance AND ≥ 80% of late-window acc): replication of Anthropic's finding in hybrid MoE reasoning. Paper-publishable as primary contribution.
- **Partial planning** (best early > 1.5× chance): suggestive, needs intervention experiments to confirm.
- **No planning** (best early ≈ chance): answer is not encoded early. Interesting negative result too.
- Bonus: gap correct > wrong → planning is causal.

In [ ]:
best_early = max([r for r in results if r['type']=='early'], key=lambda r: r['acc'])
best_late = max([r for r in results if r['type']=='late'], key=lambda r: r['acc'])

early_vs_chance = best_early['acc'] / chance
early_vs_late = best_early['acc'] / best_late['acc'] if best_late['acc'] > 0 else 0

if early_vs_chance > 2 and early_vs_late > 0.8:
    verdict = 'STRONG_PLANNING'
    emoji = '\u2b50'
    msg = f'Best early config ({best_early["name"]}) achieves {best_early["acc"]:.3f} acc ({early_vs_chance:.1f}× chance, {early_vs_late:.1%} of late-window). Anthropic-style forward planning replicated in Qwen3.6 hybrid MoE reasoning.'
elif early_vs_chance > 1.5:
    verdict = 'PARTIAL_PLANNING'
    emoji = '\u2705'
    msg = f'Suggestive evidence: {best_early["acc"]:.3f} ({early_vs_chance:.1f}× chance). Needs intervention experiments to confirm causality.'
elif early_vs_chance > 1.1:
    verdict = 'WEAK_SIGNAL'
    emoji = '\u26a0\ufe0f'
    msg = f'Marginal: {best_early["acc"]:.3f} ({early_vs_chance:.1f}× chance). Could be artifact of reasoning traces leaking letter info.'
else:
    verdict = 'NO_PLANNING'
    emoji = '\u274c'
    msg = f'Best early acc ({best_early["acc"]:.3f}) \u2248 chance ({chance:.3f}). Answer is NOT encoded in early activations. Negative result — Qwen3.6 reasoning does NOT forward-plan like Claude poetry.'

print('='*70)
print(f'  {emoji} VERDICT: {verdict}')
print('='*70)
print(msg)
print()
print(f'Best EARLY config: {best_early["name"]}  acc={best_early["acc"]:.3f}')
print(f'Best LATE config:  {best_late["name"]}  acc={best_late["acc"]:.3f}')
print(f'Chance: {chance:.3f}')
print()
print(f'Early vs chance: {early_vs_chance:.2f}×')
print(f'Early vs late (saturation): {early_vs_late:.1%}')

report = dict(
    timestamp=time.strftime('%Y-%m-%d %H:%M:%S'),
    model='Qwen/Qwen3.6-35B-A3B',
    corpus=CFG['stage_b_repo'],
    n_rollouts_analyzed=int(len(labels_letter)),
    n_classes=int(n_classes),
    chance=float(chance),
    verdict=verdict,
    message=msg,
    best_early=best_early,
    best_late=best_late,
    early_vs_chance_ratio=float(early_vs_chance),
    early_vs_late_ratio=float(early_vs_late),
    all_results=results,
    correct_vs_wrong=dict(
        acc_correct=float(acc_correct) if acc_correct else None,
        acc_wrong=float(acc_wrong) if acc_wrong else None,
    ),
    cites=dict(
        anthropic_biology='https://transformer-circuits.pub/2025/attribution-graphs/biology.html',
        look_ahead_planning='arxiv:2406.16033',
        circuit_tracer='https://github.com/safety-research/circuit-tracer',
    ),
)

with open(OUT / 'planning_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'\n\u2705 saved {OUT}/planning_report.json')

## Cell 10 — Per-discipline breakdown

Does planning depend on discipline? STEM reasoning may differ from knowledge-recall.

In [ ]:
test_disciplines = [m['discipline'] for m in meta_info]
test_disc_array = np.array(test_disciplines)

X = early_pools[w_best][li_best]
scaler = StandardScaler().fit(X[idx_tr])
pca = PCA(n_components=128, random_state=42).fit(scaler.transform(X[idx_tr]))
lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42, multi_class='multinomial').fit(
    pca.transform(scaler.transform(X[idx_tr])), y[idx_tr])

y_pred_te = lr.predict(pca.transform(scaler.transform(X[idx_te])))
y_true_te = y[idx_te]
disc_te = test_disc_array[idx_te]

per_disc = []
for disc in sorted(set(disc_te)):
    mask = disc_te == disc
    if mask.sum() < 5: continue
    acc_d = accuracy_score(y_true_te[mask], y_pred_te[mask])
    per_disc.append((disc, int(mask.sum()), acc_d))
    print(f'  {disc:30s}  n={mask.sum():3d}  acc={acc_d:.3f}  ({acc_d/chance:.1f}× chance)')

report['per_discipline'] = [dict(name=d, n=n, acc=float(a)) for d, n, a in per_disc]
with open(OUT / 'planning_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\u2705 report updated with per-discipline')